In [3]:
import pandas as pd
import glob
import os
import numpy as np
import helpers
import streamlit as st
import matplotlib.pyplot as plt
from io import BytesIO
from matplotlib.backends.backend_pdf import PdfPages


combined_df = helpers.load_data_from_game_data('Wis.-Whitewater')


# Define ALL potential columns for grouping based on your provided scouted roles
ALL_GROUPING_COLS = [
    "Driver/Shooter", 
    "Post Scorer", 
    "Playmaker", 
    "Mid-Range Specialist", 
    "Spot-Up Shooter", 
    "Rebounder", 
    "Not Scouted"
] 

# Loop through the grouping columns and convert their values to strings
if not combined_df.empty:
    for col in ALL_GROUPING_COLS:
        if col in combined_df.columns:
            # Convert values to strings
            combined_df[col] = combined_df[col].astype(str)


agg_dict = {
        'DURATION_SECONDS': 'sum',
        'TEAM_POINTS': 'sum',
        'OPPONENT_POINTS': 'sum',
        'IS_END_OF_POSSESSION': 'sum',
        'GAME_ID': 'nunique'
    }

# TEAM_NAME = "Wis.-Whitewater"
# OPPONENT_NAME = "Ripon"

# Filter out 'LINEUP' if it exists here, as it is always included
ALL_GROUPING_COLS = [col for col in ALL_GROUPING_COLS if col != 'LINEUP'] 




# Define all columns that should be kept in the final results dataframe
RATING_COLS = ['POSSESSIONS', 'Points For', 'Points Against',
               'Plus/Minus', 'Offensive Rating', 'Defensive Rating', 'Net Rating']

# --- CORE DATA CALCULATION ---


# 1. Calculate Results for TOP 3 and PDF (LINEUP ONLY)
group_dict_top = ['LINEUP']
kept_cols_top = ['LINEUP'] + RATING_COLS
away_final_results_top = helpers.calculate_lineup_ratings(combined_df, group_dict_top, agg_dict)
print(away_final_results_top)



# --- 3. Streamlit Helper Functions & Data Preparation ---

# Function to reorder columns with 'LINEUP' and grouping context first
def reorder_lineup_first(df, group_cols):
    if 'LINEUP' in df.columns:
        # Create new list with 'LINEUP' and other group cols first, followed by rating cols
        cols = [col for col in group_cols if col in df.columns] + [col for col in df.columns if col not in group_cols]
        return df[cols]
    return df

def prepare_dataframe(df, cols, group_dict_used):
    if df.empty:
        return pd.DataFrame(columns=cols)
        
    df = df[[c for c in cols if c in df.columns]].copy()
    
    # Apply the column reorder function here
    return reorder_lineup_first(df, group_dict_used)

# --- Sort helpers (Top 3 functions) ---
def top3_by_possessions(df):
    return df.sort_values(by='POSSESSIONS', ascending=False).head(3)

def top3_by_plus_minus(df):
    return df.sort_values(by=['Plus/Minus', 'POSSESSIONS'], ascending=[False, False]).head(3)

def top3_by_net_rating(df):
    return df.sort_values(by=['Net Rating', 'POSSESSIONS'], ascending=[False, False]).head(3)


# --- 4. STREAMLIT APP DISPLAY ---

# Prepare base dataframe for TOP 3 (LINEUP ONLY)
df_base_top = prepare_dataframe(away_final_results_top, kept_cols_top, group_dict_top)


# Get Top 3 DataFrames (derived from LINEUP ONLY results)
top_possessions_list = top3_by_possessions(df_base_top)
top_plus_minus_list = top3_by_plus_minus(df_base_top)
top_net_rating_list = top3_by_net_rating(df_base_top)

# --- Prepare Top 3 DataFrames for Streamlit Display ---
def prepare_top3_display_df(df_list):
    df_display = df_list.copy()
    if 'LINEUP' in df_display.columns and not df_display.empty and isinstance(df_display['LINEUP'].iloc[0], list):
        df_display['LINEUP'] = df_display['LINEUP'].apply(lambda x: ", ".join(x))
    return df_display

top_possessions_full = prepare_top3_display_df(top_possessions_list)
top_plus_minus_full = prepare_top3_display_df(top_plus_minus_list)
top_net_rating_full = prepare_top3_display_df(top_net_rating_list)

# Top 3 display uses LINEUP column only, since it was calculated with LINEUP only
POS_COLS = ['LINEUP', 'POSSESSIONS']
top_possessions = top_possessions_full[[col for col in POS_COLS if col in top_possessions_full.columns]]

top_possessions
    

--- Loading CSVs from: C:\Users\frits\OneDrive\Documents\UWW MBB\Game_Data ---
Loaded: 6510366.csv
Loaded: 6512406.csv
Loaded: 6512971.csv
Loaded: 6513259.csv
Loaded: 6513261.csv
Loaded: 6533362.csv
                                            LINEUP AGGREGATED_TIME_MM:SS  \
59       [Marino, Madson, Verges, Lukenbill, Bara]                 35:19   
146         [Madson, Verges, Bara, Rogers, Warren]                 14:11   
68          [Marino, Madson, Verges, Bara, Warren]                 11:02   
136       [Madson, Verges, Lukenbill, Quast, Bara]                 10:36   
48   [Marino, Madson, Chestnut, Verges, Lukenbill]                 06:41   
..                                             ...                   ...   
29       [Ambrose, Madson, Chestnut, Bara, Warren]                 00:00   
19      [Keyes, Turner, Wildes, Rogers, Grunewald]                 00:00   
18     [Keyes, Robinson, Turner, Grunewald, Lynch]                 00:00   
16    [Keyes, Robinson, Turner, Rogers, G

,LINEUP,POSSESSIONS
59,"Marino, Madson, Verges, Lukenbill, Bara",90
146,"Madson, Verges, Bara, Rogers, Warren",42
136,"Madson, Verges, Lukenbill, Quast, Bara",38


In [22]:
away_final_results_top['LINEUP'].iloc[0]#.sort()

['Bara', 'Lukenbill', 'Madson', 'Marino', 'Verges']